In [33]:
from tdc.multi_pred import DDI

data = DDI(name="DrugBank", path="/media/onlyzabao/Data/TDC").get_data()
print(data.shape)

Found local copy...
Loading...
Done!


(191808, 5)


In [34]:
import pandas as pd
import json

# Filter row
selectedLabel = [31, 49, 50, 56, 66, 86]
data = data[data["Y"].isin(selectedLabel)]

# Filter column
data = data[["Drug1_ID", "Drug2_ID", "Y"]]

# Filter hetionet
with open("../../Hetionet/vocab/entity_vocab.json", "r") as file:
    entity = json.load(file)

data["Drug1_ID"] = "Compound::" + data["Drug1_ID"]
data["Drug2_ID"] = "Compound::" + data["Drug2_ID"]
data = data[
    data["Drug1_ID"].isin(entity)
    & data["Drug2_ID"].isin(entity)
]

data["Interaction"] = "CaC"

print(data)

                 Drug1_ID           Drug2_ID   Y Interaction
28691   Compound::DB00968  Compound::DB00752  31         CaC
28692   Compound::DB00968  Compound::DB01037  31         CaC
28693   Compound::DB00968  Compound::DB01626  31         CaC
28694   Compound::DB00968  Compound::DB00614  31         CaC
28695   Compound::DB00968  Compound::DB01171  31         CaC
...                   ...                ...  ..         ...
191802  Compound::DB00437  Compound::DB00542  86         CaC
191803  Compound::DB00437  Compound::DB00492  86         CaC
191805  Compound::DB00437  Compound::DB00790  86         CaC
191806  Compound::DB00415  Compound::DB00437  86         CaC
191807  Compound::DB00437  Compound::DB00691  86         CaC

[46667 rows x 4 columns]


In [35]:
from sklearn.model_selection import train_test_split


def split_data(data: pd.DataFrame, train_size, test_size, val_size):
    train_data, test_data = train_test_split(data, test_size=1 - train_size)
    test_data, val_data = train_test_split(
        test_data, test_size=val_size / (test_size + val_size)
    )

    return train_data, test_data, val_data


train_data, test_data, val_data = split_data(data, 0.7, 0.2, 0.1)

print(train_data.shape)
print(test_data.shape)
print(val_data.shape)

(32666, 4)
(9334, 4)
(4667, 4)


In [43]:
def write_data(path, data: pd.DataFrame, head_index, tail_index, edge_index, inverses):
    with open(path, "w") as file:
        for _, row in data.iterrows():
            if not inverses:
                head = row[head_index]
                tail = row[tail_index]
                edge = row[edge_index]
            else:
                head = row[tail_index]
                tail = row[head_index]
                edge = "_" + row[edge_index]

            file.write(head + "\t" + edge + "\t" + tail + "\n")


files = [
    {"path": "../dev.txt", "data": val_data, "inverses": False},
    {"path": "../test.txt", "data": test_data, "inverses": False},
    {"path": "../train.txt", "data": train_data, "inverses": False},
    {"path": "../graph_triples_DB.txt", "data": train_data, "inverses": False},
    {"path": "../graph_inverses_DB.txt", "data": train_data, "inverses": True},
]

head_index = "Drug1_ID"
tail_index = "Drug2_ID"
edge_index = "Interaction"
for file in files:
    write_data(
        file["path"], file["data"], head_index, tail_index, edge_index, file["inverses"]
    )